# Data Preprocessing

Combine a session's raw camera-pose, raw EMG, and EMG packet timestamps into a model-ready dataframe `df_model` with EMG envelopes and arm joint angles sampled on the EMG timeline.

The notebook is organized as a pipeline: each stage is a pure function in its own cell, composed by `preprocess_session` at the bottom. The final cell auto-discovers all sessions in `recordings/` and runs the pipeline on each one (with a per-session prompt before overwriting existing outputs).

**Pipeline stages**

1. Load raw pose + EMG + EMG packet timestamps
2. Pose -> wide per-frame format -> 4-DOF joint angles via inverse kinematics
3. EMG ingest: uniform timeline + active-channel detection
4. Pose/EMG overlap window
5. EMG conditioning (normalize -> rectify -> lowpass -> z-score)
6. Pose alignment & upsampling to the EMG rate
7. Assemble `df_model`; 7b is an FK/IK sanity-check animation
8. Save to `training_data/training_data_{time_stamp}.csv`


In [ ]:
import os
import re
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d

from arm_inverse_kinematics import compute_joint_angles_from_data, forward_kinematics_fixed
from arm_visualizer import ArmVisualizer

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)


@dataclass
class PreprocessParams:
    # I/O
    recordings_dir:    str  = 'recordings'
    training_data_dir: str  = 'training_data'

    # Arm geometry (measured per subject; consumed by IK + FK)
    upper_arm_length_m: float = 14 * 0.0254   # 0.3556 m
    forearm_length_m:   float = 10 * 0.0254   # 0.254 m

    # EMG active-channel detection
    min_active_nonzero_fraction: float = 0.05

    # EMG / pose temporal offset (positive = EMG recorded after pose)
    emg_pose_offset_samples: int = 0

    # EMG envelope filter
    emg_lowpass_hz:    float = 4.0
    emg_lowpass_order: int   = 4

    # Arm visualization (3D FK sanity-check animation)
    viz_downsample_n: int = 400

    # Diagnostics
    display_intermediate:  bool = True
    show_sanity_animation: bool = True


## 0. Discover sessions

`discover_timestamps` returns the sorted list of timestamps in `recordings_dir` that have all three required files: `camera0_{ts}_poses_smoothed.csv`, `emg_data_{ts}.csv`, and `emg_timestamps_{ts}.csv`. Sessions missing any are skipped.


In [ ]:
def discover_timestamps(recordings_dir: str) -> list[str]:
    pose_re = re.compile(r'^camera0_(\d{8}_\d{6})_poses_smoothed\.csv$')
    files = os.listdir(recordings_dir)
    pose_ts = {m.group(1) for f in files if (m := pose_re.match(f))}
    emg_ts  = {f[len('emg_data_'):-len('.csv')]
               for f in files if f.startswith('emg_data_') and f.endswith('.csv')}
    ts_ts   = {f[len('emg_timestamps_'):-len('.csv')]
               for f in files if f.startswith('emg_timestamps_') and f.endswith('.csv')}
    return sorted(pose_ts & emg_ts & ts_ts)


## 1. Load raw recordings

Three CSVs per session, all under `recordings/`:

- `camera0_{time_stamp}_poses_smoothed.csv` -- per-frame pose detections (long format)
- `emg_data_{time_stamp}.csv` -- raw EMG channels (50 samples per packet)
- `emg_timestamps_{time_stamp}.csv` -- Unix `pc_time` for each EMG packet


In [ ]:
def load_session(time_stamp: str, params: PreprocessParams):
    d = params.recordings_dir
    pose_raw = pd.read_csv(os.path.join(d, f'camera0_{time_stamp}_poses_smoothed.csv'))
    emg_raw  = pd.read_csv(os.path.join(d, f'emg_data_{time_stamp}.csv'))
    emg_ts   = pd.read_csv(os.path.join(d, f'emg_timestamps_{time_stamp}.csv'))

    print(f"Pose shape: {pose_raw.shape}")
    print(f"EMG shape: {emg_raw.shape}")
    print(f"EMG timestamps shape: {emg_ts.shape}")

    if params.display_intermediate:
        print("\nPose head:");           display(pose_raw.head())
        print("\nEMG head:");            display(emg_raw.head())
        print("\nEMG timestamps head:"); display(emg_ts.head())

    return pose_raw, emg_raw, emg_ts


## 2. Pose -> joint angles

### 2a. Wide per-frame format

Pivot the long-format pose CSV so each frame is a single row carrying x/y/z for `shoulder`, `elbow`, and `bracelet`, then linearly interpolate across frames to fill any missing landmarks.


In [ ]:
def pose_to_wide(pose_raw: pd.DataFrame, params: PreprocessParams) -> pd.DataFrame:
    pose_wide_raw = pose_raw.pivot_table(
        index=['frame_idx', 'video_time_s'],
        columns='position_name',
        values=['x', 'y', 'z'],
        aggfunc='mean',
    )
    pose_wide_raw.columns = [f'{pos}_{coord}' for coord, pos in pose_wide_raw.columns]
    pose_wide_raw = pose_wide_raw.reset_index()

    coord_cols = [c for c in pose_wide_raw.columns if c not in ['frame_idx', 'video_time_s']]
    pose_wide = pose_wide_raw.copy()
    pose_wide[coord_cols] = pose_wide[coord_cols].interpolate(
        method='linear', limit_direction='both', axis=0,
    )

    if params.display_intermediate:
        missing_per_frame = pose_wide_raw.isna().sum(axis=1)
        print(f"Frames with missing values:\n{missing_per_frame.value_counts().sort_index()}")
        print(f"Remaining NaNs after interpolation: {pose_wide.isna().sum().sum()}")
        display(pose_wide[10:20])

    return pose_wide


### 2b. Inverse kinematics

Solve the 4-DOF shoulder/elbow joint chain for `q1..q4` from shoulder->elbow->bracelet positions and the measured upper-arm length. `compute_joint_angles_from_data` resolves the per-joint sign ambiguity by tracking the previous frame's solution.


In [ ]:
def compute_joint_angles(pose_wide: pd.DataFrame, params: PreprocessParams) -> pd.DataFrame:
    joint_angles = compute_joint_angles_from_data(pose_wide, L1=params.upper_arm_length_m)
    if params.display_intermediate:
        display(joint_angles.head())
    return joint_angles


## 3. EMG ingest

### 3a. Uniform timeline from packet timestamps

Each packet carries 50 samples and one Unix `pc_time`. Approximate per-sample timestamps as uniformly spaced between the first and last packet's `pc_time`.


In [ ]:
def emg_timeline(emg_raw: pd.DataFrame, emg_ts: pd.DataFrame):
    packet_times = emg_ts['pc_time'].values
    total_emg_samples = emg_raw.shape[0]
    sample_period = (packet_times[-1] - packet_times[0]) / (total_emg_samples - 1)
    fs_emg = 1.0 / sample_period
    t_emg  = packet_times[0] + sample_period * np.arange(total_emg_samples)

    print(f"Estimated EMG sampling rate: {fs_emg:.2f} Hz")
    print(f"EMG time range: {t_emg[0]:.3f} - {t_emg[-1]:.3f} (Unix seconds)")
    return t_emg, fs_emg, packet_times


### 3b. Active-channel detection

A channel counts as active if at least `min_active_nonzero_fraction` of its samples are non-zero. Flat-zero channels (dropped electrodes) are discarded.


In [ ]:
def detect_active_channels(emg_raw: pd.DataFrame, t_emg: np.ndarray,
                           params: PreprocessParams):
    total = emg_raw.shape[0]
    active_channels = [
        ch for ch in emg_raw.columns
        if (emg_raw[ch] != 0).sum() / total >= params.min_active_nonzero_fraction
    ]
    emg_active = emg_raw[active_channels].copy()
    emg_active['time'] = t_emg

    print(f"Active channels (>= {params.min_active_nonzero_fraction*100:.0f}% non-zero): "
          f"{active_channels}")

    if params.display_intermediate:
        display(emg_active.head())
        fig, axes = plt.subplots(len(active_channels), 1, sharex=True,
                                 figsize=(12, 2 * len(active_channels)))
        if len(active_channels) == 1:
            axes = [axes]
        for i, ch in enumerate(active_channels):
            axes[i].plot(emg_active['time'], emg_active[ch], linewidth=0.5)
            axes[i].set_ylabel(ch)
        axes[-1].set_xlabel('Time (s)')
        fig.suptitle('Raw active EMG')
        plt.tight_layout()
        plt.show()

    return emg_active, active_channels


## 4. Pose/EMG overlap window

Both streams carry their own clocks. Express each in seconds-relative-to-`session_epoch`, apply `emg_pose_offset_samples`, and compute the overlap interval `[t_min, t_max]` plus per-stream boolean masks. EMG conditioning (sec 5) and pose alignment (sec 6) both reuse these masks.


In [ ]:
def overlap_window(joint_angles: pd.DataFrame, t_emg: np.ndarray, fs_emg: float,
                   packet_times: np.ndarray, params: PreprocessParams):
    session_epoch = np.floor(packet_times[0])
    pose_time_rel = joint_angles['video_time_s'].values
    emg_time_rel  = t_emg - session_epoch - params.emg_pose_offset_samples / fs_emg

    t_min = max(pose_time_rel[0], emg_time_rel[0])
    t_max = min(pose_time_rel[-1], emg_time_rel[-1])

    mask_pose = (pose_time_rel >= t_min) & (pose_time_rel <= t_max)
    mask_emg  = (emg_time_rel  >= t_min) & (emg_time_rel  <= t_max)

    print(f"Pose time range: {pose_time_rel[0]:.3f} - {pose_time_rel[-1]:.3f} s")
    print(f"EMG  time range: {emg_time_rel[0]:.3f} - {emg_time_rel[-1]:.3f} s "
          f"(offset {params.emg_pose_offset_samples} samples = "
          f"{params.emg_pose_offset_samples / fs_emg:+.3f} s)")
    print(f"Overlap: {t_min:.3f} - {t_max:.3f} s "
          f"(pose: {mask_pose.sum()} frames, EMG: {mask_emg.sum()} samples)")

    return pose_time_rel, emg_time_rel, mask_pose, mask_emg


## 5. EMG conditioning (on trimmed signal)

Trim the active EMG to the overlap window, then per channel:

1. **Min-max normalize** to [0, 1] so channels with different gains are comparable.
2. **Rectify** (absolute value) -- EMG sign is meaningless for amplitude.
3. **Low-pass filter** (zero-phase Butterworth at `emg_lowpass_hz`).
4. **Standardize** (z-score) so each channel has mean 0, std 1.

All statistics (steps 1 and 4) are computed on the trimmed segment only.


In [ ]:
def condition_emg(emg_active: pd.DataFrame, active_channels: list[str],
                  mask_emg: np.ndarray, emg_time_rel: np.ndarray, fs_emg: float,
                  params: PreprocessParams):
    emg_trimmed      = emg_active[mask_emg].copy()
    emg_time_trimmed = emg_time_rel[mask_emg]

    emg_normalized = emg_trimmed.copy()
    for ch in active_channels:
        lo, hi = emg_normalized[ch].min(), emg_normalized[ch].max()
        emg_normalized[ch] = (emg_normalized[ch] - lo) / (hi - lo)

    emg_rectified = emg_normalized.abs()

    b, a = butter(params.emg_lowpass_order,
                  params.emg_lowpass_hz / (0.5 * fs_emg), btype='low')
    emg_filtered = pd.DataFrame(index=emg_rectified.index)
    for ch in active_channels:
        emg_filtered[ch] = filtfilt(b, a, emg_rectified[ch])
    emg_filtered['time'] = emg_time_trimmed

    emg_cols = [ch for ch in emg_filtered.columns if ch != 'time']
    emg_standardized = emg_filtered.copy()
    emg_standardized[emg_cols] = (
        (emg_filtered[emg_cols] - emg_filtered[emg_cols].mean())
        / emg_filtered[emg_cols].std()
    )

    print(f"EMG conditioned ({len(active_channels)} channels, {len(emg_trimmed)} samples)")

    if params.display_intermediate:
        ch = active_channels[0]
        fig, ax = plt.subplots(3, 1, sharex=True, figsize=(12, 8))
        ax[0].plot(emg_time_trimmed, emg_rectified[ch],   linewidth=0.5)
        ax[0].set_ylabel(f'{ch} rectified')
        ax[1].plot(emg_time_trimmed, emg_filtered[ch],    color='orange')
        ax[1].set_ylabel(f'{ch} lowpass {params.emg_lowpass_hz} Hz')
        ax[2].plot(emg_time_trimmed, emg_standardized[ch], color='green')
        ax[2].set_ylabel(f'{ch} z-score')
        ax[-1].set_xlabel('Time (s)')
        fig.suptitle(f'EMG conditioning - {ch}')
        plt.tight_layout()
        plt.show()

    return emg_standardized, emg_time_trimmed, emg_cols


## 6. Pose alignment & upsampling

Trim pose joint angles to the overlap window, then linearly interpolate `q1..q4` onto the EMG sample timestamps so pose and EMG share the same timeline at EMG resolution.


In [ ]:
def upsample_joints(joint_angles: pd.DataFrame, mask_pose: np.ndarray,
                    pose_time_rel: np.ndarray, emg_time_trimmed: np.ndarray,
                    params: PreprocessParams) -> pd.DataFrame:
    pose_trimmed      = joint_angles[mask_pose].copy()
    pose_time_trimmed = pose_time_rel[mask_pose]

    out = pd.DataFrame({'time': emg_time_trimmed})
    for col in ['q1', 'q2', 'q3', 'q4']:
        f = interp1d(pose_time_trimmed, pose_trimmed[col].values,
                     kind='linear', fill_value='extrapolate')
        out[col] = f(emg_time_trimmed)

    print(f"Joints upsampled to EMG rate: {out.shape}")

    if params.display_intermediate:
        fig, ax = plt.subplots(4, 1, sharex=True, figsize=(10, 8))
        for i, col in enumerate(['q1', 'q2', 'q3', 'q4']):
            ax[i].plot(pose_time_trimmed, pose_trimmed[col], 'o', markersize=2,
                       label='30 Hz original')
            ax[i].plot(emg_time_trimmed,  out[col], linewidth=0.8,
                       label='upsampled')
            ax[i].set_ylabel(col)
        ax[-1].set_xlabel('Time (s)')
        ax[0].legend()
        fig.suptitle('Joint angles: original vs upsampled to EMG rate')
        plt.tight_layout()
        plt.show()

    return out


## 7. Assemble model-ready dataframe

`df_model` has columns `[time, <active EMG channels>, q1, q2, q3, q4]`, one row per EMG sample. EMG channels are z-scored; joint angles are in radians.


In [ ]:
def assemble_df_model(emg_standardized: pd.DataFrame, emg_cols: list[str],
                      joint_upsampled_df: pd.DataFrame,
                      params: PreprocessParams) -> pd.DataFrame:
    df_model = pd.concat(
        [emg_standardized['time'],
         emg_standardized[emg_cols],
         joint_upsampled_df[['q1', 'q2', 'q3', 'q4']]],
        axis=1,
    )

    non_time_cols = emg_cols + ['q1', 'q2', 'q3', 'q4']
    time_diffs    = df_model['time'].diff().dropna()
    print(f"df_model shape: {df_model.shape}")
    print(f"Sample period: {time_diffs.mean():.4f} s +/- {time_diffs.std():.6f} s")

    if params.display_intermediate:
        display(df_model.head())
        print('Mean (~0 for EMG, natural for joints):')
        display(df_model[non_time_cols].mean())
        print('Std (~1 for EMG, natural for joints):')
        display(df_model[non_time_cols].std())

    return df_model


## 7b. Arm reconstruction sanity check

Animate the ground-truth shoulder->elbow->wrist trajectory from the pose CSV alongside an FK-reconstructed arm driven by the IK joints `q1..q4`. If the IK is consistent with the FK convention the two arms overlap. Restricted to the overlap window and downsampled to `viz_downsample_n` frames.


In [ ]:
def arm_sanity_animation(pose_wide: pd.DataFrame, joint_angles: pd.DataFrame,
                         mask_pose: np.ndarray, params: PreprocessParams):
    pose_viz   = pose_wide[mask_pose].reset_index(drop=True)
    joints_viz = joint_angles[mask_pose].reset_index(drop=True)

    T_full = len(pose_viz)
    step   = max(1, T_full // params.viz_downsample_n)
    idx    = np.arange(0, T_full, step)[:params.viz_downsample_n]
    pose_ds   = pose_viz.iloc[idx]
    joints_ds = joints_viz.iloc[idx]

    gt_elbow = pose_ds[['elbow_x', 'elbow_y', 'elbow_z']].values \
             - pose_ds[['shoulder_x', 'shoulder_y', 'shoulder_z']].values
    gt_wrist = pose_ds[['bracelet_x', 'bracelet_y', 'bracelet_z']].values \
             - pose_ds[['shoulder_x', 'shoulder_y', 'shoulder_z']].values

    viz = ArmVisualizer(fk_func=forward_kinematics_fixed,
                        L1=params.upper_arm_length_m,
                        L2=params.forearm_length_m)
    viz.add_arm(name='Ground truth (pose)', color='green', dash='dash',
                elbow=gt_elbow, wrist=gt_wrist)
    viz.add_arm(name='FK from IK joints', color='orange',
                q1=joints_ds['q1'].values, q2=joints_ds['q2'].values,
                q3=joints_ds['q3'].values, q4=joints_ds['q4'].values)
    viz.show(times=joints_ds['video_time_s'].values)


## 8. Save training data

Write `df_model` to `{training_data_dir}/training_data_{time_stamp}.csv`. Downstream training/eval notebooks load this file directly.


In [ ]:
def training_data_path(time_stamp: str, params: PreprocessParams) -> str:
    return os.path.join(params.training_data_dir, f'training_data_{time_stamp}.csv')


def save_training_data(df_model: pd.DataFrame, time_stamp: str,
                       params: PreprocessParams) -> str:
    os.makedirs(params.training_data_dir, exist_ok=True)
    out_path = training_data_path(time_stamp, params)
    df_model.to_csv(out_path, index=False)
    print(f'Wrote {out_path}  ({df_model.shape[0]} rows x {df_model.shape[1]} cols)')
    return out_path


## 9. Per-session pipeline

`preprocess_session(time_stamp, params)` composes every stage above. Call it directly for a single session, or via the batch driver in the next cell for many.


In [ ]:
def preprocess_session(time_stamp: str,
                       params: PreprocessParams = None) -> pd.DataFrame:
    if params is None:
        params = PreprocessParams()

    print(f"\n========== {time_stamp} ==========")
    pose_raw, emg_raw, emg_ts = load_session(time_stamp, params)
    pose_wide    = pose_to_wide(pose_raw, params)
    joint_angles = compute_joint_angles(pose_wide, params)

    t_emg, fs_emg, packet_times = emg_timeline(emg_raw, emg_ts)
    emg_active, active_channels = detect_active_channels(emg_raw, t_emg, params)

    pose_time_rel, emg_time_rel, mask_pose, mask_emg = overlap_window(
        joint_angles, t_emg, fs_emg, packet_times, params,
    )

    emg_standardized, emg_time_trimmed, emg_cols = condition_emg(
        emg_active, active_channels, mask_emg, emg_time_rel, fs_emg, params,
    )

    joint_upsampled_df = upsample_joints(
        joint_angles, mask_pose, pose_time_rel, emg_time_trimmed, params,
    )

    df_model = assemble_df_model(emg_standardized, emg_cols, joint_upsampled_df, params)

    if params.show_sanity_animation:
        arm_sanity_animation(pose_wide, joint_angles, mask_pose, params)

    save_training_data(df_model, time_stamp, params)
    return df_model


## 10. Batch driver

By default this discovers every session under `recordings/` and runs the pipeline on each one. For any timestamp whose `training_data_{ts}.csv` already exists you'll get a prompt:

- `y` -- re-run this one
- `n` -- skip this one (default)
- `a` -- re-run this and **all** subsequent already-existing ones
- `s` -- skip this and **all** subsequent already-existing ones
- `q` -- abort the batch

For a single session or a custom list, pass `timestamps=['20260425_013519']` (or any list) to `run_batch`. To tweak parameters, instantiate `PreprocessParams(...)` with overrides and pass it via `params=`.


In [ ]:
def _prompt_overwrite(path: str) -> str:
    while True:
        ans = input(f"  {path} already exists. Re-run [y/N/a=all/s=skip-all/q=quit]? ").strip().lower()
        if ans == '':
            ans = 'n'
        if ans in ('y', 'n', 'a', 's', 'q'):
            return ans
        print("  please answer y, n, a, s, or q")


def run_batch(timestamps: list[str] = None, params: PreprocessParams = None):
    if params is None:
        params = PreprocessParams()
    if timestamps is None:
        timestamps = discover_timestamps(params.recordings_dir)

    print(f"Discovered {len(timestamps)} session(s): {timestamps}")

    overwrite_mode = None  # None | 'always' | 'never'
    results = {}
    for ts in timestamps:
        out_path = training_data_path(ts, params)
        if os.path.exists(out_path):
            if overwrite_mode == 'always':
                pass
            elif overwrite_mode == 'never':
                print(f"\nSkipping {ts} (output exists)")
                continue
            else:
                ans = _prompt_overwrite(out_path)
                if ans == 'q':
                    print("Aborted.")
                    break
                if ans == 'n':
                    print(f"Skipping {ts}")
                    continue
                if ans == 's':
                    overwrite_mode = 'never'
                    print(f"Skipping {ts} and all remaining existing outputs")
                    continue
                if ans == 'a':
                    overwrite_mode = 'always'

        try:
            results[ts] = preprocess_session(ts, params)
        except Exception as e:
            print(f"!! {ts} failed: {type(e).__name__}: {e}")
            results[ts] = None

    n_ok = sum(v is not None for v in results.values())
    print(f"\n========== Batch complete: {n_ok}/{len(results)} succeeded ==========")
    return results


# Default: batch over every discovered session.
# Intermediate plots and the 3D sanity animation are off in batch mode --
# flip these on (or pass a single timestamp) for interactive single-session work.
batch_params = PreprocessParams(display_intermediate=False, show_sanity_animation=False)
results = run_batch(params=batch_params)
